# Jira Cloud Connector Sample

Load Jira Cloud issues into a Delta table: a full load on the first run, then incremental loads. Read-only against Jira. Replace all `<PLACEHOLDER>` values before running.

## Prerequisites

1. Upload `jira_client.py` (the only helper file, from this folder) to a workspace folder and set `HELPER_DIR` below to that folder.
2. Provide `JIRA_SITE` (`<site>.atlassian.net`), `JIRA_EMAIL` and `JIRA_API_TOKEN` ([create a token](https://id.atlassian.com/manage/api-tokens)) as environment variables on the cluster, or as OCI Vault secrets with `OCI_VAULT_ID` set (this also needs the `oci` package, and `OCI_COMPARTMENT_ID` if the secrets are not in the tenancy root). Do not type the token into a cell.
3. The cluster needs the `requests` package and network access to your Jira site.

## Configuration

In [ ]:
import sys

HELPER_DIR = "<WORKSPACE_PATH_TO_HELPER_FOLDER>"  # contains jira_client.py
TARGET = "<CATALOG>.<SCHEMA>.jira_issue"
JQL = "<JQL_QUERY>"  # for example: project = <PROJECT_KEY>
PAGE_SIZE = 100
OVERLAP_SECONDS = 300
sys.path.insert(0, HELPER_DIR)

# Pin the Spark session to UTC so the watermark read back from Delta is
# unambiguous; the session's local timezone would otherwise shift it.
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Read Issues

Jira interprets JQL date-time literals in the searching account's own timezone, not UTC, so the account timezone is fetched once per run and passed to the search. `search_issues` would fetch it itself if omitted; it is passed here to make the step visible.

In [ ]:
from datetime import datetime, timezone
import jira_client as j

site, email, token = j.credentials_from_env()
session = j.jira_session(email, token)
tz_name = j.account_timezone(session, site)

# Incremental: start from the newest row already loaded (None on the first run).
since = None
if spark.catalog.tableExists(TARGET):
    # Read the watermark as a UTC string: PySpark returns TIMESTAMPs in the
    # driver's local timezone, which would otherwise shift `since`.
    since = j.parse_watermark(spark.sql(
        f"SELECT date_format(max(updated), 'yyyy-MM-dd HH:mm:ss.SSS') AS m FROM {TARGET}"
    ).first()["m"])

until = datetime.now(timezone.utc)  # fixed for this run
issues = j.search_issues(
    session, site, query=JQL, tz_name=tz_name,
    since=since, until=until, overlap_seconds=OVERLAP_SECONDS, page_size=PAGE_SIZE,
)
df = j.to_dataframe(spark, list(issues))
print("issues read:", df.count())

## Write to Delta

The overlap window re-reads a few issues on purpose, so later runs `MERGE` on `key` to remove duplicates.

In [ ]:
if not spark.catalog.tableExists(TARGET):
    df.write.format("delta").saveAsTable(TARGET)
else:
    df.createOrReplaceTempView("incoming_issue")
    spark.sql(
        f"MERGE INTO {TARGET} t USING incoming_issue s ON t.key = s.key "
        "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *"
    )
print("issues in target:", spark.table(TARGET).count())

## Limits

- An `updated` watermark does not see Jira issue deletes.
- An issue updated after `until` while the read is running is picked up by the next run.
- Custom fields (`customfield_NNNNN`) are not in the typed output; they appear in the trailing `raw_fields` JSON column.
- A personal or free Jira Cloud site is for learning and testing, not production volume.
- The read materialises every issue in driver memory, so for a very large first load narrow `JQL` (for example by project or date range) and load in windows.
- The `ORDER BY` guard is a simple text check, so a query containing the phrase `order by` inside a quoted string is rejected.